# 19｜分类评价：从准确率到可解释的决策

## 从一次筛查开始 / Start with a screening decision

假设我们要从记录中找出少见的正类。若只有 10% 的记录为正，全部预测为负也有 90% 的准确率，但一个正例都找不到。本课用合成数据学习评价流程，不代表真实筛查系统的性能。

Suppose only 10% of records are positive. Predicting all negatives achieves 90% accuracy while finding no positives. This lesson uses synthetic data to practice evaluation.

**完成后你能 / Outcomes:** 读混淆矩阵，计算精确率与召回率；建立无泄漏的预处理流程；在验证集选阈值，然后在测试集报告一次结果。

In [1]:
from support import ROOT, check, show, style

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1 让四个结果有名字 / Name the four outcomes

| 真实 / Actual | 预测负 / Predicted negative | 预测正 / Predicted positive |
| --- | --- | --- |
| 负 / Negative | TN | FP |
| 正 / Positive | FN | TP |

精确率 $TP/(TP+FP)$ 回答“预测为正的样本中有多少是真的”；召回率 $TP/(TP+FN)$ 回答“真实正例中找到了多少”。分母为零时这里按 0 处理。Precision measures the fraction of predicted positives that are correct; recall measures the fraction of actual positives found.

### 练习 eval01：从计数到指标 / Metrics from counts

某模型的 TP=8、FP=2、FN=12。计算精确率 `ex_precision` 和召回率 `ex_recall`。Given TP=8, FP=2 and FN=12, compute precision and recall.

In [3]:
ex_precision=None
ex_recall=None

In [4]:
# 填写上一格后，运行检查。
check('precision',ex_precision,.8)
check('recall',ex_recall,.4)

待填写：precision
待填写：recall


<details>
<summary>提示</summary>

分别使用 TP+FP 和 TP+FN 作分母。

</details>

<details>
<summary>参考答案</summary>

```python
ex_precision=8/(8+2)
ex_recall=8/(8+12)
```

</details>

## 2 先划分，再学习统计量 / Split before fitting statistics

训练集拟合标准化参数和模型；验证集选择阈值；测试集在选择结束后评价。`Pipeline` 保证 `StandardScaler` 的均值和标准差由训练集拟合，并原样用于其他集合。

Fit preprocessing and the model on training data only. Use validation data to choose a threshold, then evaluate the selected rule on the held-out test set.

In [5]:
X, y = make_classification(n_samples=1500, n_features=8, n_informative=5,
                           n_redundant=0, weights=[.9, .1],
                           class_sep=1.0, random_state=17)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=.4, stratify=y, random_state=17)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=.5, stratify=y_temp, random_state=17)
print('train / validation / test:', len(y_train), len(y_val), len(y_test))
print('positive counts:', int(y_train.sum()), int(y_val.sum()), int(y_test.sum()))

train / validation / test: 900 300 300
positive counts: 97 32 32


### 练习 eval02：检查划分 / Check the split

将三个集合的样本数之和保存为 `ex_total`。Save the total number of examples as `ex_total`.

In [6]:
ex_total=None

In [7]:
# 填写上一格后，运行检查。
check('total',ex_total,1500)

待填写：total


<details>
<summary>提示</summary>

把三个标签数组的长度相加。

</details>

<details>
<summary>参考答案</summary>

```python
ex_total=len(y_train)+len(y_val)+len(y_test)
```

</details>

In [8]:
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=17))
model.fit(X_train, y_train)
print('baseline validation accuracy:', round(baseline.score(X_val, y_val), 3))
print('model validation accuracy:', round(model.score(X_val, y_val), 3))

baseline validation accuracy: 0.893
model validation accuracy: 0.893


## 3 在验证集选择阈值 / Select a threshold on validation data

模型给出正类概率；阈值越低，通常找到更多正例，也可能带来更多误报。我们只在预先列出的阈值中选取：要求召回率至少为 0.70，满足条件时选择精确率最高者；精确率相同时选较高阈值。若没有候选满足条件，就停止并修改方案，而不是使用测试集来选。

Choose among the predefined thresholds using validation recall ≥ 0.70, then maximize precision; break ties toward the higher threshold. If none qualifies, revise the plan without consulting the test set.

In [9]:
thresholds = [.1, .2, .3, .4, .5, .6, .7, .8, .9]
val_prob = model.predict_proba(X_val)[:, 1]
rows = []
for threshold in thresholds:
    pred = (val_prob >= threshold).astype(int)
    precision = precision_score(y_val, pred, zero_division=0)
    recall = recall_score(y_val, pred, zero_division=0)
    rows.append((threshold, precision, recall))
    print(f'threshold={threshold:.1f}  precision={precision:.3f}  recall={recall:.3f}')
eligible = [row for row in rows if row[2] >= .70]
if not eligible:
    raise ValueError('No threshold meets the validation recall target.')
chosen = max(eligible, key=lambda row: (row[1], row[0]))
print('chosen from validation:', chosen)

threshold=0.1  precision=0.241  recall=0.875
threshold=0.2  precision=0.370  recall=0.625
threshold=0.3  precision=0.571  recall=0.500
threshold=0.4  precision=0.611  recall=0.344
threshold=0.5  precision=0.500  recall=0.156
threshold=0.6  precision=0.333  recall=0.062


threshold=0.7  precision=0.667  recall=0.062


threshold=0.8  precision=1.000  recall=0.031
threshold=0.9  precision=1.000  recall=0.031
chosen from validation: (0.1, 0.2413793103448276, 0.875)


### 练习 eval03：理解阈值 / Apply a threshold

对概率 `[0.1, 0.7, 0.4]` 使用阈值 0.4，结果保存为 `ex_labels`。Apply threshold 0.4 to these probabilities and store the predicted labels.

In [10]:
ex_labels=None

In [11]:
# 填写上一格后，运行检查。
check('labels',ex_labels,[0,1,1])

待填写：labels


<details>
<summary>提示</summary>

大于或等于阈值记为 1。

</details>

<details>
<summary>参考答案</summary>

```python
ex_labels=(np.array([.1,.7,.4]) >= .4).astype(int)
```

</details>

## 4 锁定规则，查看测试集 / Lock the rule and inspect test results

以下测试结果描述固定数据划分上的一次实验。混淆矩阵按行真实类别、按列预测类别排列；报告计数与指标，能看清误报和漏报各有多少。

The matrix has actual classes in rows and predicted classes in columns. Report counts alongside metrics so false positives and false negatives remain visible.

In [12]:
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= chosen[0]).astype(int)
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
print('confusion matrix [[TN, FP], [FN, TP]]:\n', cm)
print('test precision:', round(precision_score(y_test, test_pred, zero_division=0), 3))
print('test recall:', round(recall_score(y_test, test_pred, zero_division=0), 3))
baseline_pred = baseline.predict(X_test)
print('baseline test accuracy:', round(baseline.score(X_test, y_test), 3))
print('baseline test precision:', round(precision_score(y_test, baseline_pred, zero_division=0), 3))
print('baseline test recall:', round(recall_score(y_test, baseline_pred, zero_division=0), 3))

confusion matrix [[TN, FP], [FN, TP]]:
 [[191  77]
 [  4  28]]
test precision: 0.267
test recall: 0.875
baseline test accuracy: 0.893
baseline test precision: 0.0
baseline test recall: 0.0


### 练习 eval04：读取漏报 / Read false negatives

从 `cm` 中读取 FN，保存为 `ex_fn`。Read the false-negative count from `cm`.

In [13]:
ex_fn=None

In [14]:
# 填写上一格后，运行检查。
check('false negatives',ex_fn,cm[1,0])

待填写：false negatives


<details>
<summary>提示</summary>

真实为正是第 1 行，预测为负是第 0 列。

</details>

<details>
<summary>参考答案</summary>

```python
ex_fn=cm[1,0]
```

</details>

## 5 延伸实验 / Next experiment

回到第 18 课：固定划分规则，在多个随机种子下重复整个流程，记录各次验证阈值、测试 precision / recall 和正例数量；比较阈值 0.5 与验证选择规则。不要把测试集上的最优阈值当作部署规则。

Repeat the complete process with several split seeds. Record the chosen validation threshold, test precision, recall, and positive counts; compare with a fixed 0.5 threshold. Keep the test set out of model and threshold selection.

阅读 / Read: [scikit-learn: Common pitfalls and data leakage](https://scikit-learn.org/stable/common_pitfalls.html) 与 [classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)。找出预处理与评价中哪些操作需要拟合数据。